# Lab 3.3: Subagent Isolation vs. Inline Search

**Day 1 - Topic 3**

Goal: measure what lands in the parent context window for a targeted inline search, a broad inline exploration, and the same exploration delegated to a subagent, then check when local grep beats an external index.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab3_3_subagent_isolation_vs_inline_search/start/lab3_3_subagent_isolation_vs_inline_search.ipynb)

In [ ]:
# No extra packages needed: this lab uses the Python standard library only.
# If running locally with the course venv already activated, you can skip this cell.
# This lab uses git and grep, which Colab provides.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: Build the service tree

The helper builds a 12-module service tree and a git repo in a temporary folder. Two modules define `verify_idempotency_token`, three call it, and seven only mention idempotency in a docstring. That is the noise a broad search has to read past.

In [ ]:
SYMBOL = "verify_idempotency_token"
DEFINERS = (4, 9)
CALLERS = (2, 7, 11)
MENTIONERS = (1, 3, 5, 6, 8, 10, 12)  # modules whose docstrings mention idempotency


def build_tree(root: Path) -> None:
    for i in range(1, 13):
        body = [f'"""Module {i:02d} request handler."""']
        if i in MENTIONERS:
            body[0] = f'"""Module {i:02d} request handler. Callers should be idempotent."""'
        body += [f"def helper_{i}_{j}(ctx):\n    return ctx" for j in range(35)]
        if i in DEFINERS:
            body.append(f'def {SYMBOL}(req):\n    """Check the idempotency key."""\n    return req.headers.get("X-Idempotency-Key")')
        if i in CALLERS:
            body.append(f"def handle_{i}(req):\n    return {SYMBOL}(req)")
        path = root / f"services/mod_{i:02d}/handler.py"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n\n".join(body) + "\n")
    git = ["git", "-c", "user.name=demo", "-c", "user.email=demo@example.com"]
    subprocess.run(["git", "init", "-q"], cwd=root, check=True)
    subprocess.run(["git", "add", "-A"], cwd=root, check=True)
    subprocess.run(git + ["commit", "-qm", "baseline"], cwd=root, check=True)


def sh(root: Path, cmd: str) -> str:
    return subprocess.run(cmd, shell=True, cwd=root, capture_output=True, text=True).stdout


def est_tokens(text: str) -> int:
    return max(1, len(text) // 4)


workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name).resolve() / "service_repo"
root.mkdir()
build_tree(root)
print(sh(root, "ls services"))

## Section 2: Targeted inline search

Grep for the exact symbol, then read only the files that define it. Token counts are measured from the real command output (characters divided by 4).

In [ ]:
def targeted_inline(root: Path) -> tuple[int, list[str], list[str]]:
    """Grep for the symbol, then read only the files that define it."""
    # TODO: hits = sh(root, f"grep -rn {SYMBOL} services")  (every line looks like "path:line:text")
    # TODO: defs = sorted set of the file paths (text before the first ":") for hit lines containing f"def {SYMBOL}"
    # TODO: tokens = est_tokens(hits) plus est_tokens(sh(root, f"cat {path}")) for each path in defs
    # TODO: callers = sorted set of the file paths for hit lines that do NOT contain f"def {SYMBOL}"
    return 0, [], []

In [ ]:
t_tokens, defs, callers = targeted_inline(root)
print(f"Targeted inline search: {t_tokens:,d} tokens")
print(f"  defined in: {defs}")
print(f"  called from: {callers}")
# Expected: 823 tokens; defined in mod_04 and mod_09; called from mod_02, mod_07, mod_11
check("Targeted inline search costs 823 tokens", t_tokens == 823)
check("Two defining modules found", defs == ["services/mod_04/handler.py", "services/mod_09/handler.py"])
check("Three calling modules found", callers == ["services/mod_02/handler.py", "services/mod_07/handler.py", "services/mod_11/handler.py"])

## Section 3: Broad exploration and the subagent model

The broad search is provided: it reads every module that mentions the topic. Then model delegation. No Claude Code subagent is launched here. The child window receives the same real tool output, and the parent receives only a short report.

In [ ]:
def broad_exploration(root: Path) -> tuple[int, list[str], list[str]]:
    """Grep the broad topic, read every matching module in full, then grep the symbol."""
    topic_files = sh(root, "grep -rl idempotent services; grep -rl idempotency services").split()
    topic_files = sorted(set(topic_files))
    tokens = est_tokens("\n".join(topic_files))
    for p in topic_files:
        tokens += est_tokens(sh(root, f"cat {p}"))
    hits = sh(root, f"grep -rn {SYMBOL} services")
    tokens += est_tokens(hits)
    defs = sorted({l.split(":")[0] for l in hits.splitlines() if f"def {SYMBOL}" in l})
    callers = sorted({l.split(":")[0] for l in hits.splitlines() if f"def {SYMBOL}" not in l})
    return tokens, defs, callers


def local_hits(root: Path, pattern: str) -> int:
    return len(sh(root, f"grep -rn {pattern} . --include=*.py --include=*.md").splitlines())

In [ ]:
def subagent_footprint(broad_tokens: int, report: str) -> tuple[int, int, int]:
    """Model delegation: return (parent window, subagent window, total model work) in tokens."""
    # TODO: The subagent reads everything the broad inline search reads, so its window is broad_tokens
    # TODO: The parent window receives only the short report, so its size is est_tokens(report)
    # TODO: Total model work is the parent window plus the subagent window
    return 0, 0, 0

In [ ]:
b_tokens, _, _ = broad_exploration(root)
report = (
    f"`{SYMBOL}` is defined in {', '.join(defs)} and called from {', '.join(callers)}. "
    "Static scan only; run pytest on those modules to confirm runtime behavior."
)
parent, child, total = subagent_footprint(b_tokens, report)

print(f"{'Mode':<50} | {'Parent window':>13} | {'Subagent window':>15} | {'Total model work':>16}")
print("-" * 104)
rows = [
    ("A. Inline, targeted (grep symbol + read 2 files)", t_tokens, 0, t_tokens),
    ("B. Inline, broad (read every module that matches)", b_tokens, 0, b_tokens),
    ("C. Subagent, same broad exploration (modeled)", parent, child, total),
]
for name, p, c, t in rows:
    print(f"{name:<50} | {p:>13,d} | {c:>15,d} | {t:>16,d}")
print(f"\nSubagent report returned to the parent ({parent} tokens): {report}")

# Expected:
# A. ... |   823 |     0 |   823
# B. ... | 4,404 |     0 | 4,404
# C. ... |    67 | 4,404 | 4,471
print()
check("Broad inline exploration costs 4,404 tokens", b_tokens == 4404)
check("The subagent's parent window is only 67 tokens", parent == 67)
check("The subagent still does 4,471 tokens of total model work", total == 4471)
check("Delegation keeps the parent smaller than even the targeted inline search", 0 < parent < t_tokens)

## Section 4: Local grep vs. an external index

Three tasks, three answers. This cell runs the real checks on the repo.

In [ ]:
# Scenario 1: exact symbol lookup.
n = local_hits(root, SYMBOL)
print(f"Task: callers of `{SYMBOL}` in this repo\n  grep hits: {n} -> local Grep + Read")

# Scenario 2: an uncommitted change vs. an index built before the edit.
index = {str(p.relative_to(root)): p.read_text() for p in root.rglob("*.py")}  # snapshot at index time
target = root / "services/mod_04/handler.py"
target.write_text(target.read_text() + "\ndef verify_idempotency_token_v2(req):\n    return None\n")
index_hits = sum(1 for text in index.values() if "verify_idempotency_token_v2" in text)
grep_hits = local_hits(root, "verify_idempotency_token_v2")
diff = sh(root, "git diff --stat").strip().splitlines()[-1]
print("Task: find an uncommitted edit made after the index was built")
print(f"  index snapshot hits: {index_hits}; grep hits: {grep_hits}; git diff: {diff}")
print("  -> working tree changes are visible to Grep and git diff, not to an index built earlier")

# Scenario 3: an external corpus.
pm = local_hits(root, "-i postmortem")
print("Task: search 4,000 historical postmortems across 15 teams")
print(f"  grep hits in this repo: {pm} -> corpus is not local; use an access-controlled retrieval service (not run here)")
workdir.cleanup()

print()
check("Scenario 1: grep finds all 5 references to the symbol", n == 5)
check("Scenario 2: the stale index misses the new edit, grep and git diff see it", (index_hits, grep_hits) == (0, 1))
check("Scenario 3: the postmortem corpus is not on disk", pm == 0)

In [ ]:
print("Lab complete.")
print("Takeaway: Delegate broad exploratory reads to a subagent so the parent keeps a short report; use direct Grep and git for local code; reserve an external index for corpora that are not on disk.")